# 🔢 Mathematical Operator Symbol Classification from Handwritten Images

**Course:** Predictive Analytics  
**Team Members:** Adithyan | Jia | Theertha  
**Topic:** Topic 34 — Mathematical Operator Symbol Classification  

---

## ⚠️ Important: Run Order
1. **Run the Dataset Generator cell first** (generates all 500 images)
2. Then run each stage in order
3. Do NOT restart runtime between stages (data stays in memory)

---

| Stage | Description | Owner |
|-------|-------------|-------|
| Dataset | Dataset Generator | Adithyan |
| 1 | Problem Definition & Literature Review | Adithyan |
| 2 | Data Collection & Understanding | Adithyan |
| 3 | Preprocessing & Cleaning | Jia |
| 4 | Exploratory Data Analysis | Adithyan |
| 5 | Feature Engineering & Selection | Theertha |
| 6 | Model Building & Training | Jia |
| 7 | Model Evaluation & Comparison | Theertha |
| 8 | Model Interpretation & Explainability | Adithyan |
| 9 | Deployment (Streamlit) | Jia |
| 10 | Documentation | Theertha |

---

In [1]:
# ============================================================
# DATASET GENERATOR — Run this FIRST before any other stage
# Generates 500 handwritten-style math operator images
# Author: Adithyan | Run once per Colab session
# ============================================================

!pip install pillow -q

import numpy as np
import os
from PIL import Image, ImageDraw
import random
import math

SIZE = 100
SAMPLES_PER_CLASS = 50
OUTPUT_DIR = '/content/dataset'
random.seed(42)
np.random.seed(42)

CLASSES = [
    'plus', 'minus', 'multiply', 'divide', 'equal',
    'not_equal', 'less_than', 'greater_than', 'plus_minus', 'sqrt'
]

def rnd(a, b):
    return random.uniform(a, b)

def rnd_int(a, b):
    return random.randint(a, b)

def make_canvas():
    bg = rnd_int(248, 255)
    img = Image.new('RGB', (SIZE, SIZE), (bg, bg, bg))
    draw = ImageDraw.Draw(img)
    for _ in range(30):
        x = rnd_int(0, SIZE-1)
        y = rnd_int(0, SIZE-1)
        a = rnd_int(2, 8)
        draw.point((x, y), fill=(255-a, 255-a, 255-a))
    return img, draw

def wobbly_line(draw, x1, y1, x2, y2, lw, wob, color):
    steps = 14
    pts = []
    for i in range(steps + 1):
        t = i / steps
        nx = rnd(-wob, wob)
        ny = rnd(-wob, wob)
        pressure = 0.75 + 0.5 * math.sin(t * math.pi)
        pts.append((x1 + (x2-x1)*t + nx, y1 + (y2-y1)*t + ny, max(1, lw*pressure)))
    for i in range(len(pts) - 1):
        xa, ya, wa = pts[i]
        xb, yb, wb = pts[i+1]
        draw.line([(xa, ya), (xb, yb)], fill=color, width=max(1, int((wa+wb)/2)))

def apply_transform(x, y, cx, cy, angle, scale, offx, offy):
    x -= cx
    y -= cy
    xr = x * math.cos(angle) - y * math.sin(angle)
    yr = x * math.sin(angle) + y * math.cos(angle)
    return xr * scale + cx + offx, yr * scale + cy + offy

def tp(pts, cx, cy, angle, scale, offx, offy):
    return [apply_transform(x, y, cx, cy, angle, scale, offx, offy) for x, y in pts]

def get_transform():
    return rnd(-0.12, 0.12), rnd(0.88, 1.10), rnd(-5, 5), rnd(-5, 5)

def get_style():
    lw = rnd(4.0, 6.5)
    wob = rnd(1.0, 2.2)
    d = rnd_int(0, 20)
    return lw, wob, (d, d, d)

def hl(draw, x1, y1, x2, y2, lw, wob, color, cx, cy, angle, scale, offx, offy):
    pts = tp([(x1, y1), (x2, y2)], cx, cy, angle, scale, offx, offy)
    wobbly_line(draw, pts[0][0], pts[0][1], pts[1][0], pts[1][1], lw, wob, color)

def draw_plus(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(23, 28)
    hl(draw, cx-a, cy, cx+a, cy, lw, wob, color, cx, cy, angle, scale, offx, offy)
    hl(draw, cx, cy-a, cx, cy+a, lw, wob, color, cx, cy, angle, scale, offx, offy)

def draw_minus(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(23, 28)
    hl(draw, cx-a, cy, cx+a, cy, lw, wob, color, cx, cy, angle, scale, offx, offy)

def draw_multiply(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(20, 25)
    hl(draw, cx-a, cy-a, cx+a, cy+a, lw, wob, color, cx, cy, angle, scale, offx, offy)
    hl(draw, cx+a, cy-a, cx-a, cy+a, lw, wob, color, cx, cy, angle, scale, offx, offy)

def draw_divide(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(23, 27)
    hl(draw, cx-a, cy, cx+a, cy, lw, wob, color, cx, cy, angle, scale, offx, offy)
    dr = rnd_int(3, 5)
    for dy in [-rnd(12, 15), rnd(12, 15)]:
        px, py = apply_transform(cx, cy+dy, cx, cy, angle, scale, offx, offy)
        draw.ellipse([(px-dr, py-dr), (px+dr, py+dr)], fill=color)

def draw_equal(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(22, 27)
    g = rnd(9, 12)
    for dy in [-g, g]:
        hl(draw, cx-a, cy+dy, cx+a, cy+dy, lw, wob, color, cx, cy, angle, scale, offx, offy)

def draw_not_equal(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(22, 27)
    g = rnd(9, 12)
    for dy in [-g, g]:
        hl(draw, cx-a, cy+dy, cx+a, cy+dy, lw, wob, color, cx, cy, angle, scale, offx, offy)
    sl = rnd(15, 18)
    hl(draw, cx+sl, cy-23, cx-sl, cy+23, lw*0.9, wob*0.8, color, cx, cy, angle, scale, offx, offy)

def draw_less_than(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(22, 26)
    tx = cx - a + rnd(3, 6)
    pts = tp([(cx+a, cy-a), (tx, cy), (cx+a, cy+a)], cx, cy, angle, scale, offx, offy)
    wobbly_line(draw, pts[0][0], pts[0][1], pts[1][0], pts[1][1], lw, wob, color)
    wobbly_line(draw, pts[1][0], pts[1][1], pts[2][0], pts[2][1], lw, wob, color)

def draw_greater_than(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(22, 26)
    tx = cx + a - rnd(3, 6)
    pts = tp([(cx-a, cy-a), (tx, cy), (cx-a, cy+a)], cx, cy, angle, scale, offx, offy)
    wobbly_line(draw, pts[0][0], pts[0][1], pts[1][0], pts[1][1], lw, wob, color)
    wobbly_line(draw, pts[1][0], pts[1][1], pts[2][0], pts[2][1], lw, wob, color)

def draw_plus_minus(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    a = rnd(20, 25)
    sh = rnd(8, 12)
    hl(draw, cx-a, cy-sh, cx+a, cy-sh, lw, wob, color, cx, cy, angle, scale, offx, offy)
    hl(draw, cx, cy-sh-a+5, cx, cy-sh+a-5, lw, wob, color, cx, cy, angle, scale, offx, offy)
    bot = cy + sh + rnd(5, 9)
    hl(draw, cx-a, bot, cx+a, bot, lw, wob, color, cx, cy, angle, scale, offx, offy)

def draw_sqrt(draw, cx, cy, lw, wob, color, angle, scale, offx, offy):
    top_y = cy - rnd(18, 22)
    raw = [
        (cx-33, cy+rnd(0, 3)),
        (cx-21, cy+rnd(13, 17)),
        (cx-7,  top_y),
        (cx+10, top_y),
        (cx+32, top_y)
    ]
    pts = tp(raw, cx, cy, angle, scale, offx, offy)
    for i in range(len(pts) - 1):
        wobbly_line(draw, pts[i][0], pts[i][1], pts[i+1][0], pts[i+1][1], lw, wob*0.8, color)

DRAWERS = {
    'plus'         : draw_plus,
    'minus'        : draw_minus,
    'multiply'     : draw_multiply,
    'divide'       : draw_divide,
    'equal'        : draw_equal,
    'not_equal'    : draw_not_equal,
    'less_than'    : draw_less_than,
    'greater_than' : draw_greater_than,
    'plus_minus'   : draw_plus_minus,
    'sqrt'         : draw_sqrt
}

# Generate all images
os.makedirs(OUTPUT_DIR, exist_ok=True)
total = 0

for cls in CLASSES:
    cls_dir = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(cls_dir, exist_ok=True)
    for i in range(SAMPLES_PER_CLASS):
        img, draw = make_canvas()
        cx, cy = SIZE/2, SIZE/2
        lw, wob, color = get_style()
        angle, scale, offx, offy = get_transform()
        DRAWERS[cls](draw, cx, cy, lw, wob, color, angle, scale, offx, offy)
        fname = cls + "_" + str(i+1).zfill(3) + ".png"
        img.save(os.path.join(cls_dir, fname))
        total += 1
    print("Done: " + cls + " - " + str(SAMPLES_PER_CLASS) + " images")

print("Dataset ready! " + str(total) + " total images in " + OUTPUT_DIR)
print("Folders: " + str(os.listdir(OUTPUT_DIR)))

# Variables used by all stages
dataset_path = OUTPUT_DIR
class_mapping = {
    'plus'         : '+',
    'minus'        : '-',
    'multiply'     : 'x',
    'divide'       : '/',
    'equal'        : '=',
    'not_equal'    : '!=',
    'less_than'    : '<',
    'greater_than' : '>',
    'plus_minus'   : '+-',
    'sqrt'         : 'sqrt'
}
print("dataset_path and class_mapping ready!")

Done: plus - 50 images
Done: minus - 50 images
Done: multiply - 50 images
Done: divide - 50 images
Done: equal - 50 images
Done: not_equal - 50 images
Done: less_than - 50 images
Done: greater_than - 50 images
Done: plus_minus - 50 images
Done: sqrt - 50 images
Dataset ready! 500 total images in /content/dataset
Folders: ['multiply', 'equal', 'less_than', 'not_equal', 'minus', 'divide', 'greater_than', 'sqrt', 'plus_minus', 'plus']
dataset_path and class_mapping ready!
